In [ ]:
!pip install tensorflow
# Import necessary libraries
import os
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam

In [ ]:
# Check if dataset directories exist
train_dir = "/content/drive/MyDrive/soybean/train"
test_dir = "/content/drive/MyDrive/soybean/test"


In [ ]:
if not os.path.exists(train_dir):
    raise FileNotFoundError(f"Training directory '{train_dir}' not found!")
if not os.path.exists(test_dir):
    raise FileNotFoundError(f"Testing directory '{test_dir}' not found!")

In [ ]:
# Load VGG16 as base model (pretrained on ImageNet)
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Freeze all layers except the last few for fine-tuning
for layer in base_model.layers[:-4]:  # Fine-tune last 4 layers
    layer.trainable = False

In [ ]:
# Data Preparation
img_size = (224, 224)
batch_size = 16

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255,  # Corrected rescale value
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Train and validation generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 3527 images belonging to 5 classes.
Found 881 images belonging to 5 classes.


In [ ]:
test_datagen = ImageDataGenerator(rescale=1.0/255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 1105 images belonging to 5 classes.


In [ ]:
# Ensure classes are detected
num_classes = len(train_generator.class_indices)  # Get the actual number of classes

In [ ]:
# Custom classification head
x = Flatten()(base_model.output)
x = Dense(1024, activation="relu")(x)
x = Dropout(0.5)(x)
output_layer = Dense(num_classes, activation="softmax")(x)  # Update output layer to match num_classes

In [ ]:
# Create model
vgg_model = Model(inputs=base_model.input, outputs=output_layer)
vgg_model.compile(optimizer=Adam(learning_rate=0.0001), loss="categorical_crossentropy", metrics=["accuracy"])


In [ ]:
# Callbacks
checkpoint = ModelCheckpoint("vgg16_best_model.h5", monitor="val_accuracy", save_best_only=True, mode="max")
early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

In [ ]:

# Train the model
history = vgg_model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    callbacks=[checkpoint, early_stopping]
)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.4589 - loss: 1.3004

221/221 ━━━━━━━━━━━━━━━━━━━━ 1496s 7s/step - accuracy: 0.4594 - loss: 1.2993 - val_accuracy: 0.7151 - val_loss: 0.7407
Epoch 2/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - accuracy: 0.7001 - loss: 0.7229

221/221 ━━━━━━━━━━━━━━━━━━━━ 79s 353ms/step - accuracy: 0.7001 - loss: 0.7229 - val_accuracy: 0.7230 - val_loss: 0.6907
Epoch 3/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step - accuracy: 0.7530 - loss: 0.6210

221/221 ━━━━━━━━━━━━━━━━━━━━ 79s 346ms/step - accuracy: 0.7530 - loss: 0.6210 - val_accuracy: 0.7480 - val_loss: 0.6223
Epoch 4/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - accuracy: 0.7683 - loss: 0.5971

221/221 ━━━━━━━━━━━━━━━━━━━━ 81s 366ms/step - accuracy: 0.7684 - loss: 0.5970 - val_accuracy: 0.7753 - val_loss: 0.6013
Epoch 5/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step - accuracy: 0.7942 - loss: 0.5290

221/221 ━━━━━━━━━━━━━━━━━━━━ 78s 347ms/step - accuracy: 0.7943 - loss: 0.5289 - val_accuracy: 0.8400 - val_loss: 0.4222
Epoch 6/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 79s 337ms/step - accuracy: 0.8225 - loss: 0.4599 - val_accuracy: 0.8150 - val_loss: 0.5305
Epoch 7/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step - accuracy: 0.8251 - loss: 0.4557

221/221 ━━━━━━━━━━━━━━━━━━━━ 75s 340ms/step - accuracy: 0.8251 - loss: 0.4556 - val_accuracy: 0.8434 - val_loss: 0.4707
Epoch 8/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 0.8692 - loss: 0.3706

221/221 ━━━━━━━━━━━━━━━━━━━━ 83s 344ms/step - accuracy: 0.8692 - loss: 0.3706 - val_accuracy: 0.8456 - val_loss: 0.4025
Epoch 9/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 74s 335ms/step - accuracy: 0.8717 - loss: 0.3433 - val_accuracy: 0.8116 - val_loss: 0.5397
Epoch 10/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 82s 339ms/step - accuracy: 0.8760 - loss: 0.3451 - val_accuracy: 0.8309 - val_loss: 0.4920


In [ ]:
# Evaluate the model
loss, accuracy = vgg_model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy:.2f}")

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


70/70 ━━━━━━━━━━━━━━━━━━━━ 444s 6s/step - accuracy: 0.8434 - loss: 0.4137
Test Accuracy: 0.87


In [ ]:
# Save the final model
vgg_model.save("vgg16_model.h5")

In [ ]:
y_true = test_generator.classes  # True labels
y_pred = vgg_model.predict(test_generator).argmax(axis=1)

70/70 ━━━━━━━━━━━━━━━━━━━━ 8s 109ms/step


In [ ]:
# Print classification report
from sklearn.metrics import classification_report # Import the classification_report function
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=validation_generator.class_indices.keys()))

Classification Report:
                       precision    recall  f1-score   support

      Broken soybeans       0.84      0.86      0.85       201
    Immature soybeans       0.91      0.75      0.82       225
      Intact soybeans       0.79      0.93      0.85       241
Skin-damaged soybeans       0.93      0.90      0.91       226
     Spotted soybeans       0.89      0.91      0.90       212

             accuracy                           0.87      1105
            macro avg       0.87      0.87      0.87      1105
         weighted avg       0.87      0.87      0.87      1105

